In [1]:
import pandas as pd
from pathlib import Path

h1 = pd.read_csv("data/raw/H1.csv")
h2 = pd.read_csv("data/raw/H2.csv")

print("H1:", h1.shape)
print("H2:", h2.shape)

display(h1.head())
display(h2.head())

H1: (40060, 31)
H2: (79330, 31)


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
0,0,342,2015,July,27,1,0,0,2,0,...,No Deposit,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,0,737,2015,July,27,1,0,0,2,0,...,No Deposit,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,0,7,2015,July,27,1,0,1,1,0,...,No Deposit,NULL,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,0,13,2015,July,27,1,0,1,1,0,...,No Deposit,304,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,0,14,2015,July,27,1,0,2,2,0,...,No Deposit,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
0,0,6,2015,July,27,1,0,2,1,0.0,...,No Deposit,6,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-03
1,1,88,2015,July,27,1,0,4,2,0.0,...,No Deposit,9,NULL,0,Transient,76.5,0,1,Canceled,2015-07-01
2,1,65,2015,July,27,1,0,4,1,0.0,...,No Deposit,9,NULL,0,Transient,68.0,0,1,Canceled,2015-04-30
3,1,92,2015,July,27,1,2,4,2,0.0,...,No Deposit,9,NULL,0,Transient,76.5,0,2,Canceled,2015-06-23
4,1,100,2015,July,27,2,0,2,2,0.0,...,No Deposit,9,NULL,0,Transient,76.5,0,1,Canceled,2015-04-02


In [2]:
# Combine the two hotel booking datasets
h1["HotelType"] = "Resort Hotel"
h2["HotelType"] = "City Hotel"

df = pd.concat([h1, h2], ignore_index=True)

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

display(df.head())

Total rows: 119390
Total columns: 32


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate,HotelType
0,0,342,2015,July,27,1,0,0,2,0.0,...,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01,Resort Hotel
1,0,737,2015,July,27,1,0,0,2,0.0,...,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01,Resort Hotel
2,0,7,2015,July,27,1,0,1,1,0.0,...,NULL,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02,Resort Hotel
3,0,13,2015,July,27,1,0,1,1,0.0,...,304,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02,Resort Hotel
4,0,14,2015,July,27,1,0,2,2,0.0,...,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03,Resort Hotel


In [3]:
# Display column names and data types
print("Column names and data types:")
print(df.dtypes)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check identical rows
print("\nIdentical rows:")
print(df.duplicated().sum())

Column names and data types:
IsCanceled                       int64
LeadTime                         int64
ArrivalDateYear                  int64
ArrivalDateMonth                   str
ArrivalDateWeekNumber            int64
ArrivalDateDayOfMonth            int64
StaysInWeekendNights             int64
StaysInWeekNights                int64
Adults                           int64
Children                       float64
Babies                           int64
Meal                               str
Country                            str
MarketSegment                      str
DistributionChannel                str
IsRepeatedGuest                  int64
PreviousCancellations            int64
PreviousBookingsNotCanceled      int64
ReservedRoomType                   str
AssignedRoomType                   str
BookingChanges                   int64
DepositType                        str
Agent                              str
Company                            str
DaysInWaitingList                in

## Task 1: Dataset Understanding and Analytical Questions

### Dataset Information
- **Dataset:** Hotel Booking Demand Dataset
- **Source:** Supplementary data from the instructor-recommended research article
- **Article URL:** https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/
- **Files:** H1.csv and H2.csv
- **Format:** CSV
- **Total source records:** 119,390
- **Original columns:** 31 per file
- **Source grain:** One row represents one recorded hotel booking observation.
- **Candidate key:** No explicit booking ID is provided. A composite key using source file and original row number will be used to identify source records.

### Four Analytical Questions
1. How do cancellation rates compare between Resort Hotels and City Hotels?
2. How do average daily rates (ADR) change across arrival months for each hotel type?
3. Which market segments have the highest numbers of bookings?
4. How does the average lead time differ between canceled and non-canceled bookings across hotel types?

### Why This Dataset Is Interesting
This dataset contains individual hotel booking observations, allowing us to investigate booking behavior, cancellation patterns, room prices, and market segments. Unlike pre-aggregated statistics, it provides opportunities to validate records, handle missing values, and construct a dimensional model for analytical queries.

In [4]:
# Data dictionary for important analytical columns

data_dictionary = pd.DataFrame({
    "Column": [
        "HotelType",
        "IsCanceled",
        "LeadTime",
        "ArrivalDateYear",
        "ArrivalDateMonth",
        "MarketSegment",
        "ADR",
        "Adults",
        "Children",
        "Babies"
    ],
    "Expected Type": [
        "string", "integer", "integer", "integer",
        "string", "string", "float",
        "integer", "float", "integer"
    ],
    "Unit / Meaning": [
        "Hotel category: Resort or City",
        "Cancellation indicator: 1 = canceled, 0 = not canceled",
        "Days between booking entry and arrival",
        "Scheduled arrival year",
        "Scheduled arrival month",
        "Booking market segment",
        "Average daily room rate (monetary units)",
        "Number of adults",
        "Number of children",
        "Number of babies"
    ]
})

display(data_dictionary)

,Column,Expected Type,Unit / Meaning
0,HotelType,string,Hotel category: Resort or City
1,IsCanceled,integer,"Cancellation indicator: 1 = canceled, 0 = not ..."
2,LeadTime,integer,Days between booking entry and arrival
3,ArrivalDateYear,integer,Scheduled arrival year
4,ArrivalDateMonth,string,Scheduled arrival month
5,MarketSegment,string,Booking market segment
6,ADR,float,Average daily room rate (monetary units)
7,Adults,integer,Number of adults
8,Children,float,Number of children
9,Babies,integer,Number of babies


In [5]:
# Missing-value summary

missing_report = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

display(
    missing_report[
        missing_report["Missing Count"] > 0
    ].sort_values("Missing Count", ascending=False)
)

,Missing Count,Missing Percentage
Country,488,0.41
Children,4,0.00


## Task 2: Ingestion Strategy

**Ingestion approach:** Initial full load followed by incremental batch ingestion for future updates.

**Batch vs. Streaming:** Batch ingestion is appropriate because H1.csv and H2.csv are historical hotel booking datasets rather than continuously arriving real-time events.

**Pull vs. Push:** Pull-based ingestion is used because the pipeline reads the source CSV files from the local project directory.

**Ingestion frequency:** The initial historical dataset is loaded once. For future updates, a monthly batch schedule is proposed if new source files become available.

**Watermark:** The source file name and file checksum will be used to identify previously processed batches. The original row number within each file will identify individual source observations.

**Recovery and reruns:** The pipeline will record each load using a load ID. Reprocessing an unchanged file should not create duplicate Bronze records. Failed loads can be retried after checking the ingestion log.

**Justification:** This approach is suitable for historical booking data because it supports reproducibility, source traceability, and controlled ingestion without requiring real-time streaming infrastructure.